In [2]:
pip install guppylang


Note: you may need to restart the kernel to use updated packages.


In [23]:
from guppylang import guppy
from guppylang.std.quantum import qubit, h, x, cx, measure
import time

In [25]:
from collections import Counter

@guppy
def oracle_11(q0: qubit, q1: qubit) -> None:
    cz(q0, q1)
    
@guppy
def diffusion(q0: qubit, q1: qubit) -> None:
    h(q0); h(q1)
    x(q0); x(q1)
    cz(q0, q1)
    x(q0); x(q1)
    h(q0); h(q1)
    
@guppy
def grover() -> None:
    q0 = qubit()
    q1 = qubit()
    h(q0)
    h(q1)
    oracle_11(q0, q1)
    diffusion(q0, q1)
    r0 = measure(q0)
    r1 = measure(q1)
    result("grover_q0", r0)
    result("grover_q1", r1)

shots = 124
bits = []

# Crear el simulador una sola vez
sim = grover.emulator(n_qubits=2).stabilizer_sim()

# Medir tiempo de ejecución
start_time = time.time()

# Ejecutar múltiples shots
for i in range(shots):
    r = sim.with_seed(i).run()
    d = dict(r.results[0].entries)
    bitstring = f"{d['grover_q0']}{d['grover_q1']}"
    bits.append(bitstring)

end_time = time.time()
elapsed_time = end_time - start_time

# Contar resultados
counts = Counter(bits)
print("Measurement counts:")
print(counts)

# Mostrar tiempo de ejecución
print(f"Tiempo de ejecución para {shots} shots: {elapsed_time:.6f} segundos")


Measurement counts:
Counter({'11': 124})
Tiempo de ejecución para 124 shots: 3.834177 segundos


In [27]:
import random
def apply_depolarizing_bitflip(bit, p):

    if random.random() < p:
        return '1' if bit == '0' else '0'
    return bit

p_1q = 0.02   # ruido en puertas de 1 qubit
p_2q = 0.05   # ruido en puertas de 2 qubits

shots = 1024
counts = Counter()
start = time.time()
for i in range(shots):

    result = grover.run()   
    
    d = dict(result.results[0].entries)
    b0 = d["q0"]
    b1 = d["q1"]
    b0 = apply_depolarizing_bitflip(b0, p_1q)
    b1 = apply_depolarizing_bitflip(b1, p_1q)
    if random.random() < p_2q:
        b0 = '1' if b0 == '0' else '0'
        b1 = '1' if b1 == '0' else '0'
    counts[f"{b0}{b1}"] += 1

end = time.time()
print("Measurement counts with simulated noise:", counts)
print(f"Tiempo total: {end - start:.6f} s")


Traceback (most recent call last):
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\AppData\Local\Temp\ipykernel_18304\1330188308.py", line 16, in <module>
    result = grover.run()
             ^^^^^^^^^^
AttributeError: 'GuppyFunctionDefinition' object has no attribute 'run'
